In [6]:
import os
import mne
import numpy as np
from mne.decoding import CSP
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

mne.set_log_level('WARNING')

In [9]:
def subject_hand_classifier(subjects, tmin=0, tmax=2, n_csp=2, test_size=0.3, random_state=42):
    hand_runs = ['R03', 'R04', 'R07', 'R08', 'R11', 'R12']  # strictly hand runs
    event_id = {'T1': 1, 'T2': 2}  # T1=left hand, T2=right hand
    results = {}
    
    for subj in subjects:
        print(f"\nProcessing {subj}...")
        subj_files = [f"{subj}/{f}" for f in os.listdir(subj) 
                      if f.endswith('.edf') and any(run in f for run in hand_runs)]
        
        X_subj, y_subj = [], []
        for fpath in subj_files:
            raw = mne.io.read_raw_edf(fpath, preload=True, verbose=False)
            events, _ = mne.events_from_annotations(raw)
            # Only keep T1 and T2 events
            mask = np.isin(events[:, -1], [1, 2])
            if not mask.any():
                continue
            epochs = mne.Epochs(raw, events[mask], event_id=event_id,
                                tmin=tmin, tmax=tmax, baseline=None,
                                preload=True, verbose=False)
            X_subj.append(epochs.get_data())
            y_subj.append(epochs.events[:, -1] - 1)  # map 1→0 (left), 2→1 (right)
        
        if not X_subj:
            print(f"No valid hand runs for {subj}, skipping.")
            continue
        
        X_subj = np.concatenate(X_subj, axis=0)
        y_subj = np.concatenate(y_subj, axis=0)
        
        csp = CSP(n_components=n_csp, reg=None, log=True, norm_trace=False)
        X_csp = csp.fit_transform(X_subj, y_subj)
        X_scaled = StandardScaler().fit_transform(X_csp)
        
        X_train, X_test, y_train, y_test = train_test_split(
            X_scaled, y_subj, test_size=test_size,
            random_state=random_state, stratify=y_subj
        )
        
        clf = LogisticRegression(max_iter=1000)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        
        acc = accuracy_score(y_test, y_pred)
        cm = confusion_matrix(y_test, y_pred)
        print(f"{subj} Hand Test Accuracy: {acc:.3f}")
        print(f"{subj} Hand Confusion Matrix:\n{cm}")
        results[subj] = {"accuracy": acc, "confusion_matrix": cm}
    
    return results

def subject_both_classifier(subjects, tmin=0, tmax=2, n_csp=2, test_size=0.3, random_state=42):
    foot_runs = ['R05', 'R06', 'R09', 'R10', 'R13', 'R14']  # strictly foot runs
    event_id = {'T1': 1, 'T2': 2}  # T1=both fists, T2=both feet
    results = {}
    
    for subj in subjects:
        print(f"\nProcessing {subj}...")
        subj_files = [f"{subj}/{f}" for f in os.listdir(subj) 
                      if f.endswith('.edf') and any(run in f for run in foot_runs)]
        
        X_subj, y_subj = [], []
        for fpath in subj_files:
            raw = mne.io.read_raw_edf(fpath, preload=True, verbose=False)
            events, _ = mne.events_from_annotations(raw, event_id=event_id)
            # Only keep T1 and T2 events
            mask = np.isin(events[:, -1], [1, 2])
            if not mask.any():
                continue
            epochs = mne.Epochs(raw, events[mask], event_id=event_id,
                                tmin=tmin, tmax=tmax, baseline=None,
                                preload=True, verbose=False)
            X_subj.append(epochs.get_data())
            y_subj.append(epochs.events[:, -1] - 1)  # map 1→0 (fists), 2→1 (feet)
        
        if not X_subj:
            print(f"No valid foot runs for {subj}, skipping.")
            continue
        
        X_subj = np.concatenate(X_subj, axis=0)
        y_subj = np.concatenate(y_subj, axis=0)
        
        csp = CSP(n_components=n_csp, reg=None, log=True, norm_trace=False)
        X_csp = csp.fit_transform(X_subj, y_subj)
        X_scaled = StandardScaler().fit_transform(X_csp)
        
        X_train, X_test, y_train, y_test = train_test_split(
            X_scaled, y_subj, test_size=test_size,
            random_state=random_state, stratify=y_subj
        )
        
        clf = LogisticRegression(max_iter=1000)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        
        acc = accuracy_score(y_test, y_pred)
        cm = confusion_matrix(y_test, y_pred)
        print(f"{subj} Foot Test Accuracy: {acc:.3f}")
        print(f"{subj} Foot Confusion Matrix:\n{cm}")
        results[subj] = {"accuracy": acc, "confusion_matrix": cm}
    
    return results

In [10]:
hand_test_results = subject_hand_classifier(['S004'])
both_test_results = subject_both_classifier(['S004'])


Processing S004...
S004 Hand Test Accuracy: 0.976
S004 Hand Confusion Matrix:
[[27  0]
 [ 1 13]]

Processing S004...
S004 Foot Test Accuracy: 0.889
S004 Foot Confusion Matrix:
[[13  0]
 [ 3 11]]
